In [1]:
"""
Train/test near-duplicate check for the Hair Diseases dataset.

Run this as a single Kaggle cell BEFORE picking a final model.
It answers one question: do augmented siblings of the same source photo
appear in both train/ and test/?

Two independent tests:
  A) Perceptual hash (dHash) -- catches exact and lightly-edited duplicates.
  B) Embedding cosine similarity -- catches rotate/zoom/shift/flip siblings
     that dHash misses.

Interpretation:
  A high match rate (>10-15%) means the reported 100% test accuracy is
  measuring memorisation of source images, not generalisation.
"""

import os
import numpy as np
from PIL import Image
from collections import defaultdict

TRAIN_DIR = '/kaggle/input/datasets/sundarannamalai/hair-diseases/Hair Diseases - Final/train'
TEST_DIR = '/kaggle/input/datasets/sundarannamalai/hair-diseases/Hair Diseases - Final/test'
IMAGE_SIZE = 224


def list_images(root):
    paths, labels = [], []
    for cls in sorted(os.listdir(root)):
        cdir = os.path.join(root, cls)
        if not os.path.isdir(cdir):
            continue
        for f in sorted(os.listdir(cdir)):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                paths.append(os.path.join(cdir, f))
                labels.append(cls)
    return paths, labels


# ---------------------------------------------------------------------
# TEST A: perceptual hash (dHash, 8x8 -> 64-bit)
# ---------------------------------------------------------------------
def dhash(path, size=8):
    img = Image.open(path).convert('L').resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(img, dtype=np.int16)
    return ''.join('1' if b else '0' for b in (a[:, 1:] > a[:, :-1]).flatten())


def hamming(h1, h2):
    return sum(c1 != c2 for c1, c2 in zip(h1, h2))


train_paths, train_labels = list_images(TRAIN_DIR)
test_paths, test_labels = list_images(TEST_DIR)
print(f'train images: {len(train_paths)}   test images: {len(test_paths)}')

print('\n--- TEST A: perceptual hash ---')
train_hashes = defaultdict(list)
for p, l in zip(train_paths, train_labels):
    train_hashes[dhash(p)].append((p, l))

exact, near, same_class = 0, 0, 0
train_hash_list = list(train_hashes.keys())
for p, l in zip(test_paths, test_labels):
    h = dhash(p)
    if h in train_hashes:
        exact += 1
        if any(tl == l for _, tl in train_hashes[h]):
            same_class += 1
    else:
        # near-duplicate: Hamming distance <= 5 out of 64 bits
        if any(hamming(h, th) <= 5 for th in train_hash_list):
            near += 1

n = len(test_paths)
print(f'exact hash matches in train : {exact}/{n}  ({exact / n:.1%})')
print(f'  ...of which same class    : {same_class}')
print(f'near matches (Hamming <= 5) : {near}/{n}  ({near / n:.1%})')
print(f'TOTAL leaked (A)            : {(exact + near) / n:.1%}')


# ---------------------------------------------------------------------
# TEST B: embedding similarity (catches augmented siblings)
# ---------------------------------------------------------------------
print('\n--- TEST B: embedding cosine similarity ---')
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

encoder = MobileNetV2(weights='imagenet', include_top=False,
                      pooling='avg', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))


def embed(paths, batch=64):
    out = []
    for i in range(0, len(paths), batch):
        imgs = np.stack([
            np.asarray(Image.open(p).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE)),
                       dtype=np.float32)
            for p in paths[i:i + batch]
        ])
        out.append(encoder.predict(preprocess_input(imgs), verbose=0))
    e = np.concatenate(out)
    return e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-9)


emb_train = embed(train_paths)
emb_test = embed(test_paths)

THRESHOLD = 0.95  # cosine sim above this = almost certainly the same source photo
best_sim = np.zeros(len(test_paths), dtype=np.float32)
best_idx = np.zeros(len(test_paths), dtype=np.int64)
for i in range(0, len(test_paths), 256):
    sims = emb_test[i:i + 256] @ emb_train.T
    best_sim[i:i + 256] = sims.max(axis=1)
    best_idx[i:i + 256] = sims.argmax(axis=1)

leaked = best_sim >= THRESHOLD
print(f'test images with a train neighbour >= {THRESHOLD} cosine: '
      f'{leaked.sum()}/{n}  ({leaked.mean():.1%})')
print(f'median nearest-neighbour similarity: {np.median(best_sim):.4f}')
print(f'mean   nearest-neighbour similarity: {best_sim.mean():.4f}')

# Per-class breakdown
print('\nper-class leak rate (Test B):')
for cls in sorted(set(test_labels)):
    m = np.array([l == cls for l in test_labels])
    print(f'  {cls:<26} {leaked[m].mean():>6.1%}   (n={m.sum()})')

# Show the worst offenders so you can eyeball them
print('\ntop 5 most-similar test/train pairs:')
for k in np.argsort(-best_sim)[:5]:
    print(f'  sim={best_sim[k]:.4f}')
    print(f'    test : {os.path.basename(test_paths[k])}  [{test_labels[k]}]')
    print(f'    train: {os.path.basename(train_paths[best_idx[k]])}  '
          f'[{train_labels[best_idx[k]]}]')

train images: 9600   test images: 1200

--- TEST A: perceptual hash ---
exact hash matches in train : 3/1200  (0.2%)
  ...of which same class    : 3
near matches (Hamming <= 5) : 291/1200  (24.2%)
TOTAL leaked (A)            : 24.5%

--- TEST B: embedding cosine similarity ---


I0000 00:00:1785156861.537198      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


I0000 00:00:1785156874.054530     126 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


test images with a train neighbour >= 0.95 cosine: 360/1200  (30.0%)
median nearest-neighbour similarity: 0.9410
mean   nearest-neighbour similarity: 0.9371

per-class leak rate (Test B):
  Alopecia Areata             35.8%   (n=120)
  Contact Dermatitis          16.7%   (n=120)
  Folliculitis                38.3%   (n=120)
  Head Lice                   22.5%   (n=120)
  Lichen Planus               30.0%   (n=120)
  Male Pattern Baldness       56.7%   (n=120)
  Psoriasis                   29.2%   (n=120)
  Seborrheic Dermatitis       28.3%   (n=120)
  Telogen Effluvium           36.7%   (n=120)
  Tinea Capitis                5.8%   (n=120)

top 5 most-similar test/train pairs:
  sim=0.9830
    test : male_pattern_baldness_0141.jpg  [Male Pattern Baldness]
    train: male_pattern_baldness_1005.jpg  [Male Pattern Baldness]
  sim=0.9811
    test : alopecia_areata_0088.jpg  [Alopecia Areata]
    train: alopecia_areata_0581.jpg  [Alopecia Areata]
  sim=0.9804
    test : lichen_planus_0944.j